# Smartphone ML Intelligence

## Part 9 — Final Price and Performance Models

### Objective

Notebooks 03 and 04 compared several models for predicting **price** and **AnTuTu performance**. This notebook:

1. selects the best-performing model from each notebook
2. re-checks their cross-validated errors with exactly the same setup
3. trains them on all phones
4. saves the models together with the information a user interface needs (typical values for skipped inputs, dropdown options, processor details, camera patterns)

The saved models power the **Price Predictor** and **Performance Predictor** pages of the Streamlit app.

### 1. Importing & Loading

In [1]:
from datetime import date
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import duckdb
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.model_selection import RepeatedKFold, GroupKFold, cross_validate

RANDOM_STATE = 42

In [2]:
DATA_PATH = Path("../data/processed/phones_clean.csv")

data = pd.read_csv(
    DATA_PATH,
    parse_dates=["launch_date"]
)

con = duckdb.connect()
con.register("phones", data)

query = """
SELECT *
FROM phones
"""

phones = con.execute(query).df()

print(f"Rows: {phones.shape[0]}")
print(f"Columns: {phones.shape[1]}")

Rows: 289
Columns: 48


As in Notebooks 03 and 04, all phones are used for modelling, not only the currently available ones: the relationship between specifications and price or performance does not depend on whether a phone is still on sale.

### 2. Choosing the Models

The cross-validated results reported in Notebooks 03 and 04 (repeated 5-fold cross-validation, 5 repeats):

**Price (Notebook 03)**

| Model | Features | MAE (Rs.) | RMSE (Rs.) | R² |
|---|---|---|---|---|
| Linear Regression | specifications | 8,910 | 14,619 | 0.823 |
| Ridge | specifications | 7,582 | 11,250 | 0.898 |
| Random Forest | specifications | 6,893 | 11,185 | 0.898 |
| Extra Trees | specifications | 6,615 | 10,848 | 0.903 |
| Log-price Gradient Boosting | specifications | 6,495 | 10,569 | 0.911 |
| **Gradient Boosting** | **specifications** | **6,526** | **10,041** | **0.918** |
| Gradient Boosting | specifications + benchmarks | 6,401 | 9,708 | 0.923 |

**Performance (Notebook 04), target: AnTuTu score**

| Model | Features | MAE | RMSE | R² |
|---|---|---|---|---|
| Ridge | hardware | 251,237 | 370,868 | 0.832 |
| Random Forest | hardware | 172,567 | 268,267 | 0.908 |
| Gradient Boosting | hardware | 183,250 | 262,349 | 0.913 |
| **Extra Trees** | **hardware** | **150,236** | **249,394** | **0.921** |
| **Extra Trees** | **hardware + Geekbench** | **107,703** | **170,318** | **0.964** |

#### Decisions

- **Price: Gradient Boosting on specifications.** It has the lowest RMSE and highest R² of the specification-only models. Log-price Gradient Boosting is essentially tied on MAE (Rs. 31 lower) but clearly worse on RMSE and R². The benchmark-enhanced model is marginally better, but it would require users to know AnTuTu and Geekbench scores, and Notebook 03 concluded that benchmarks add little beyond the specifications. This is also the model used for the value score in Notebook 06.
- **Performance: Extra Trees, in two versions.**
    - *Hardware only* — the lowest-error model that needs only specifications.
    - *Hardware + Geekbench* — clearly more accurate, used whenever the user knows both Geekbench scores.

  Filling unknown Geekbench scores with a median would throw away that advantage and mislead the model, so the app switches models instead.

### 3. Price Model

The features, preprocessing, and hyperparameters are exactly those of the specification-only Gradient Boosting model in Notebook 03.

In [3]:
price_excluded = [
    "price",
    "name",
    "launch_date",
    "colors",
    "camera_sensor_name",
    "processor"
]

benchmark_features = [
    "antutu",
    "geekbench_single",
    "geekbench_multiple"
]

price_features = [
    column
    for column in phones.columns
    if column not in price_excluded + benchmark_features
]

X_price = phones[price_features].copy()
y_price = phones["price"].copy()

price_numeric = X_price.select_dtypes(include=["number", "bool"]).columns.tolist()
price_categorical = X_price.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print(f"Price features: {len(price_features)}")
print("Numeric:", price_numeric)
print("Categorical:", price_categorical)

Price features: 39
Numeric: ['fab', 'size', 'curved_display', 'refresh_rate', 'hbm', 'ppi', 'battery', 'charging_speed', 'thickness', 'weight', 'camera_number', 'useful_camera_number', 'main', 'ultrawide', 'third_mp', 'fourth_mp', 'front', 'wireless_charging', 'ip_rating', 'availability', 'has_ultrawide', 'has_third_camera', 'has_fourth_camera', 'launch_year', 'phone_age_days']
Categorical: ['brand', 'sale', 'processor_type', 'ram_type', 'rom_type', 'display_type', 'protection', 'back_material', 'frame_material', 'third', 'fourth', 'rear_video', 'front_video', 'speaker']


In [4]:
price_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                transformers=[
                    (
                        "numeric",
                        Pipeline([
                            ("imputer", SimpleImputer(strategy="median")),
                            ("scaler", StandardScaler())
                        ]),
                        price_numeric
                    ),
                    (
                        "categorical",
                        Pipeline([
                            ("imputer", SimpleImputer(strategy="most_frequent")),
                            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
                        ]),
                        price_categorical
                    )
                ]
            )
        ),
        (
            "model",
            GradientBoostingRegressor(
                n_estimators=300,
                learning_rate=0.03,
                max_depth=2,
                min_samples_leaf=3,
                random_state=RANDOM_STATE
            )
        )
    ]
)

In [5]:
cv = RepeatedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=RANDOM_STATE
)

scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}


def summarise(results):
    return {
        "mae": float(-results["test_mae"].mean()),
        "rmse": float(-results["test_rmse"].mean()),
        "r2": float(results["test_r2"].mean())
    }


price_cv = summarise(
    cross_validate(price_pipeline, X_price, y_price, cv=cv, scoring=scoring, n_jobs=-1)
)

print(f"Price model MAE:  Rs. {price_cv['mae']:,.0f}")
print(f"Price model RMSE: Rs. {price_cv['rmse']:,.0f}")
print(f"Price model R²:   {price_cv['r2']:.3f}")

TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

### 4. Performance Models

Both are the Extra Trees models of Notebook 04 with the same features and settings. Besides the standard cross-validation, the **processor-grouped** validation from Notebook 04 is repeated: it measures the error for a processor the model has never seen, which is the realistic error when a user's processor is not in the dataset.

In [ ]:
hardware_features = ["processor", "processor_type", "fab", "ram_type", "rom_type", "size", "thickness"]
geekbench_features = hardware_features + ["geekbench_single", "geekbench_multiple"]

performance_categorical = ["processor", "processor_type", "ram_type", "rom_type"]


def extra_trees_pipeline(numeric_features):

    return Pipeline(
        steps=[
            (
                "preprocessor",
                ColumnTransformer(
                    transformers=[
                        ("numeric", Pipeline([("scaler", StandardScaler())]), numeric_features),
                        ("categorical", Pipeline([("encoder", OneHotEncoder(handle_unknown="ignore"))]),
                         performance_categorical)
                    ]
                )
            ),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=500,
                    min_samples_leaf=2,
                    random_state=RANDOM_STATE,
                    n_jobs=-1
                )
            )
        ]
    )


hardware_pipeline = extra_trees_pipeline(["fab", "size", "thickness"])
geekbench_pipeline = extra_trees_pipeline(["fab", "size", "thickness", "geekbench_single", "geekbench_multiple"])

y_performance = phones["antutu"].copy()
groups = phones["processor"]
group_cv = GroupKFold(n_splits=5)

performance_cv = {}

for name, pipeline, features in [
    ("hardware", hardware_pipeline, hardware_features),
    ("geekbench", geekbench_pipeline, geekbench_features),
]:
    X = phones[features].copy()

    performance_cv[name] = {
        "cv": summarise(cross_validate(pipeline, X, y_performance, cv=cv, scoring=scoring, n_jobs=-1)),
        "grouped_cv": summarise(cross_validate(pipeline, X, y_performance, cv=group_cv, groups=groups,
                                               scoring=scoring, n_jobs=-1))
    }

pd.DataFrame({
    f"{name} — {kind}": values
    for name, results in performance_cv.items()
    for kind, values in results.items()
}).T.round({"mae": 0, "rmse": 0, "r2": 3})

,mae,rmse,r2
hardware — cv,150570.0,249238.0,0.921
hardware — grouped_cv,272469.0,369129.0,0.812
geekbench — cv,108180.0,170764.0,0.963
geekbench — grouped_cv,169355.0,240883.0,0.919


### 5. Reproduced vs Original Results

The re-computed errors should match the values reported in Notebooks 03 and 04, confirming that the same models were rebuilt.

In [ ]:
reported = {
    "Price — Gradient Boosting (Notebook 03)": {"mae": 6526, "rmse": 10041, "r2": 0.918},
    "Performance — Extra Trees, hardware (Notebook 04)": {"mae": 150236, "rmse": 249394, "r2": 0.921},
    "Performance — Extra Trees, + Geekbench (Notebook 04)": {"mae": 107703, "rmse": 170318, "r2": 0.964},
    "Performance — hardware, processor-grouped (Notebook 04)": {"mae": 273659, "rmse": 369701, "r2": 0.811},
    "Performance — + Geekbench, processor-grouped (Notebook 04)": {"mae": 170381, "rmse": 240831, "r2": 0.919},
}

reproduced = [
    price_cv,
    performance_cv["hardware"]["cv"],
    performance_cv["geekbench"]["cv"],
    performance_cv["hardware"]["grouped_cv"],
    performance_cv["geekbench"]["grouped_cv"],
]

comparison = pd.DataFrame([
    {
        "model": name,
        "reported MAE": original["mae"],
        "reproduced MAE": round(new["mae"]),
        "reported R²": original["r2"],
        "reproduced R²": round(new["r2"], 3),
    }
    for (name, original), new in zip(reported.items(), reproduced)
])

comparison

,model,reported MAE,reproduced MAE,reported R²,reproduced R²
0,Price — Gradient Boosting (Notebook 03),6526,6538,0.918,0.918
1,"Performance — Extra Trees, hardware (Notebook 04)",150236,150570,0.921,0.921
2,"Performance — Extra Trees, + Geekbench (Notebo...",107703,108180,0.964,0.963
3,"Performance — hardware, processor-grouped (Not...",273659,272469,0.811,0.812
4,"Performance — + Geekbench, processor-grouped (...",170381,169355,0.919,0.919


Small differences can appear if the processed dataset was regenerated after Notebooks 03 and 04 were run (for example, phone ages are counted from the day Notebook 01 was run, and the dataset is maintained over time). The processor-grouped results also depend on how the processors are assigned to the five folds, which can differ between scikit-learn versions. The re-computed values are the ones saved with the models.

### 6. Training the Final Models

Cross-validation estimated how well each model generalises. The final models are now trained on **all** phones, so the app uses every available example.

In [ ]:
price_model = price_pipeline.fit(X_price, y_price)

hardware_model = hardware_pipeline.fit(phones[hardware_features], y_performance)
geekbench_model = geekbench_pipeline.fit(phones[geekbench_features], y_performance)

print("All three models trained on", len(phones), "phones.")

All three models trained on 289 phones.


### 7. Information for the App

The app asks for as few inputs as possible, and every input can be skipped. This section prepares what that needs.

#### 7.1 Typical values for skipped inputs

A skipped number is replaced by the **median** of all phones, and a skipped category by the **mode** (most common value).

In [ ]:
PRICE_SIMPLE_INPUTS = [
    "brand", "sale", "ram_type", "rom_type", "display_type", "size", "curved_display",
    "refresh_rate", "hbm", "protection", "ppi", "battery", "charging_speed", "thickness",
    "back_material", "frame_material", "weight", "rear_video", "front", "front_video",
    "speaker", "wireless_charging", "ip_rating"
]

INPUT_LABELS = {
    "brand": "Brand",
    "sale": "Sales channel / tier",
    "processor_type": "Processor type",
    "fab": "Fabrication (nm)",
    "ram_type": "RAM type",
    "rom_type": "Storage type",
    "display_type": "Display type",
    "size": "Screen size (inches)",
    "curved_display": "Curved display",
    "refresh_rate": "Refresh rate (Hz)",
    "hbm": "Peak brightness (nits)",
    "protection": "Screen protection",
    "ppi": "Pixel density (ppi)",
    "battery": "Battery (mAh)",
    "charging_speed": "Charging speed (W)",
    "thickness": "Thickness (mm)",
    "back_material": "Back material",
    "frame_material": "Frame material",
    "weight": "Weight (g)",
    "rear_video": "Rear video",
    "front": "Front camera (MP)",
    "front_video": "Front video",
    "speaker": "Speakers",
    "wireless_charging": "Wireless charging",
    "ip_rating": "IP rating",
    "useful_camera_number": "Useful cameras",
    "main": "Main camera (MP)",
}


def to_python(value):
    """Convert NumPy values to plain Python values for JSON."""
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if np.isnan(value) else float(value)
    if pd.isna(value):
        return None
    return value


def typical_value(series):
    """Median for numbers, mode for categories and yes/no values."""
    if pd.api.types.is_bool_dtype(series) or not pd.api.types.is_numeric_dtype(series):
        return to_python(series.mode().iloc[0])
    median = series.median()
    if pd.api.types.is_integer_dtype(series):
        return int(round(median))
    return round(float(median), 2)


default_columns = PRICE_SIMPLE_INPUTS + ["processor_type", "fab", "useful_camera_number", "main"]

defaults = {column: typical_value(phones[column]) for column in default_columns}

pd.DataFrame({
    "input": [INPUT_LABELS[column] for column in default_columns],
    "value used when skipped": [defaults[column] for column in default_columns],
})

,input,value used when skipped
0,Brand,Oppo
1,Sales channel / tier,Online
2,RAM type,LPDDR4X
3,Storage type,UFS 2.2
4,Display type,Amoled
5,Screen size (inches),6.74
6,Curved display,False
7,Refresh rate (Hz),120
8,Peak brightness (nits),1300.0
9,Screen protection,"Corning, GG 7i"


#### 7.2 Dropdown options

Every category, and every number with only a few distinct values (refresh rate, IP rating, camera megapixels), becomes a dropdown with the values that occur in the dataset. Choosing from a list is faster than typing, and it guarantees values the model has seen.

In [ ]:
DROPDOWN_COLUMNS = [
    "brand", "sale", "processor_type", "fab", "ram_type", "rom_type", "display_type", "refresh_rate",
    "protection", "back_material", "frame_material", "rear_video", "front", "front_video", "speaker",
    "ip_rating", "useful_camera_number", "main"
]

options = {
    column: sorted(to_python(value) for value in phones[column].dropna().unique())
    for column in DROPDOWN_COLUMNS
}

options["ultrawide"] = sorted(to_python(v) for v in phones["ultrawide"].dropna().unique())

ranges = {
    column: [to_python(phones[column].min()), to_python(phones[column].max())]
    for column in ["size", "hbm", "ppi", "battery", "charging_speed", "thickness", "weight"]
}

for column in ["brand", "sale", "refresh_rate", "ip_rating", "main"]:
    print(f"{column:<12} {options[column]}")

brand        ['Apple', 'Google', 'IQOO', 'Infinix', 'Lava', 'Motorola', 'Nothing', 'OnePlus', 'Oppo', 'POCO', 'Realme', 'Samsung', 'Tecno', 'Vivo', 'Xiaomi']
sale         ['Both', 'Flagship', 'Offline', 'Online']
refresh_rate [60, 90, 120, 144, 165]
ip_rating    [52, 53, 54, 64, 65, 66, 67, 68, 69]
main         [8, 13, 32, 48, 50, 64, 108, 200]


#### 7.3 Processors

Users pick a processor by name, for example "Snapdragon 6 Gen 3". Its processor type and fabrication size then follow automatically, so they do not have to be entered.

The processor also makes skipped values more realistic. Filling a skipped RAM type with the overall mode (LPDDR4X, typical of budget phones) next to a flagship chip would describe a phone that does not exist and mislead the models. So when a processor is chosen, skipped **RAM type, storage type, screen size, and thickness** are filled with the mode or median **among phones with that processor**. Without a processor, the overall values from Section 7.1 are used.

In [ ]:
processor_table = (
    phones
    .groupby(["processor_type", "processor"])["fab"]
    .agg(lambda s: s.mode().iloc[0])
    .reset_index()
    .sort_values(["processor_type", "processor"])
)

PROCESSOR_TYPICAL_COLUMNS = ["ram_type", "rom_type", "size", "thickness"]

processors = []

for row in processor_table.itertuples():

    same_processor = phones[
        (phones["processor_type"] == row.processor_type)
        & (phones["processor"] == row.processor)
    ]

    processors.append({
        "label": f"{row.processor_type} {row.processor}",
        "processor": row.processor,
        "processor_type": row.processor_type,
        "fab": int(row.fab),
        "phones": len(same_processor),
        "typical": {column: typical_value(same_processor[column]) for column in PROCESSOR_TYPICAL_COLUMNS},
    })

print(f"Processors: {len(processors)}")

pd.DataFrame([
    {"processor": p["label"], "fab": p["fab"], "phones": p["phones"], **p["typical"]}
    for p in processors
]).iloc[[0, 7, 30, 60]]

Processors: 79


,processor,fab,phones,ram_type,rom_type,size,thickness
0,Apple Bionic A16,4,2,LPDDR5,NVMe,6.4,7.8
7,Exynos 1330,5,4,LPDDR4X,UFS 2.2,6.7,7.5
30,Mediatek Dimensity 8350,4,5,LPDDR5X,UFS 4.0,6.7,8.2
60,Snapdragon 782G,6,1,LPDDR4X,UFS 3.1,6.7,8.2


#### 7.4 Camera patterns

The price model uses eleven camera columns. The app asks only for:

| Useful cameras | Asked |
|---|---|
| 1 | main camera |
| 2 | main camera, and whether the second camera is an ultrawide (its megapixels if so, otherwise the telephoto zoom) |
| 3 | main camera, ultrawide, telephoto zoom |
| 4 | main camera, ultrawide, telephoto zoom |

A zoom of 3.5 means `Telephoto - 3.5x`, the format used in the dataset.

Everything else, such as the total camera count, extra low-resolution lenses (depth or macro), the telephoto's megapixels, and the second telephoto of 4-camera phones, is taken from the **most common pattern among phones with the same number of useful cameras**:

In [ ]:
def is_telephoto(label):
    return isinstance(label, str) and label.startswith("Telephoto")


telephoto_zoom = phones["third"].where(phones["third"].map(is_telephoto)).str.extract(r"(\d+(?:\.\d+)?)x")[0].astype(float)

camera_patterns = {}

for useful in sorted(phones["useful_camera_number"].unique()):

    group = phones[phones["useful_camera_number"] == useful]

    if useful == 2:
        # Most 2-camera phones pair the main camera with an ultrawide
        group = group[group["has_ultrawide"]]

    columns = ["camera_number", "third", "third_mp", "fourth", "fourth_mp"]
    most_common = (
        group[columns]
        .astype(object)
        .where(group[columns].notna(), "<none>")
        .value_counts()
        .index[0]
    )

    camera_patterns[str(useful)] = {
        column: (None if value == "<none>" else to_python(value))
        for column, value in zip(columns, most_common)
    }

camera_metadata = {
    "patterns": camera_patterns,
    "ultrawide_mp_default": to_python(phones.loc[phones["has_ultrawide"], "ultrawide"].median()),
    "telephoto_mp_default": to_python(phones.loc[phones["third"].map(is_telephoto), "third_mp"].median()),
    "zoom_default": to_python(telephoto_zoom.median()),
    "zoom_options": sorted(to_python(v) for v in telephoto_zoom.dropna().unique()),
}

pd.DataFrame(camera_patterns).T.rename_axis("useful cameras")

,camera_number,third,third_mp,fourth,fourth_mp
useful cameras,,,,,
1,2,Useless,2.0,None,None
2,2.0,NaN,NaN,NaN,NaN
3,3,Telephoto - 3x,50.0,None,None
4,4,Telephoto - 5x,50.0,Telephoto - 3x,10.0


In [ ]:
print("Ultrawide MP when skipped :", camera_metadata["ultrawide_mp_default"])
print("Telephoto MP (always)     :", camera_metadata["telephoto_mp_default"])
print("Zoom when skipped         :", camera_metadata["zoom_default"])
print("Zoom options              :", camera_metadata["zoom_options"])

Ultrawide MP when skipped : 8.0
Telephoto MP (always)     : 50.0
Zoom when skipped         : 3.0
Zoom options              : [2.0, 2.5, 2.6, 3.0, 3.2, 3.5, 3.7, 4.0, 4.3, 5.0, 6.0]


### 8. Saving

The models and the metadata are written to the `models/` folder at the project root. The `.gitignore` of this project excludes `models/`, so the files stay local; anyone cloning the repository creates them by running this notebook.

The models must be loaded with the same scikit-learn version that trained them, which is automatically the case on the same machine.

In [ ]:
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(price_model, MODELS_DIR / "price_model.joblib")
joblib.dump(hardware_model, MODELS_DIR / "performance_hardware_model.joblib")
joblib.dump(geekbench_model, MODELS_DIR / "performance_geekbench_model.joblib")

metadata = {
    "created": date.today().isoformat(),
    "phones_used": len(phones),
    "price": {
        "name": "Gradient Boosting (specifications)",
        "features": price_features,
        "simple_inputs": PRICE_SIMPLE_INPUTS,
        "cv": price_cv,
    },
    "performance": {
        "hardware": {
            "name": "Extra Trees (hardware)",
            "features": hardware_features,
            **performance_cv["hardware"],
        },
        "geekbench": {
            "name": "Extra Trees (hardware + Geekbench)",
            "features": geekbench_features,
            **performance_cv["geekbench"],
        },
    },
    "defaults": defaults,
    "input_labels": INPUT_LABELS,
    "options": options,
    "ranges": ranges,
    "processors": processors,
    "camera": camera_metadata,
}

(MODELS_DIR / "app_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

for file in sorted(MODELS_DIR.iterdir()):
    print(f"{file.name:<40} {file.stat().st_size / 1024:>8.0f} KB")

app_metadata.json                              32 KB
performance_geekbench_model.joblib           8410 KB
performance_hardware_model.joblib            7799 KB
price_model.joblib                            221 KB


### 9. Using the Saved Models

The app turns user inputs into model inputs with `app/prediction_models.py`. Importing it here checks the saved files and shows the input logic at work.

In [ ]:
sys.path.append(str(Path("../app").resolve()))

import prediction_models as pm

models, saved_metadata = pm.load_assets(MODELS_DIR)

print("Loaded:", ", ".join(models))

Loaded: price, performance_hardware, performance_geekbench


#### 9.1 Everything skipped

With every input skipped, the price model receives a phone made of typical values. The prediction should be in the middle of the market.

In [ ]:
typical = pm.predict_price({}, models, saved_metadata)

print(f"Predicted price for a typical phone: Rs. {typical['prediction']:,.0f}")
print(f"Median price in the dataset:         Rs. {phones['price'].median():,.0f}")

Predicted price for a typical phone: Rs. 30,541
Median price in the dataset:         Rs. 32,000


#### 9.2 The camera rule

How the few camera inputs become the model's camera columns:

In [ ]:
camera_examples = [
    ("1 camera", dict(useful_cameras=1, main_mp=50, has_ultrawide=None, ultrawide_mp=None, zoom=None)),
    ("2 cameras, ultrawide", dict(useful_cameras=2, main_mp=50, has_ultrawide=True, ultrawide_mp=8, zoom=None)),
    ("2 cameras, no ultrawide", dict(useful_cameras=2, main_mp=50, has_ultrawide=False, ultrawide_mp=None, zoom=2)),
    ("3 cameras", dict(useful_cameras=3, main_mp=50, has_ultrawide=None, ultrawide_mp=50, zoom=3.5)),
    ("4 cameras", dict(useful_cameras=4, main_mp=200, has_ultrawide=None, ultrawide_mp=50, zoom=5)),
]

pd.DataFrame({
    label: pm.camera_fields(metadata=saved_metadata, **inputs)[0]
    for label, inputs in camera_examples
}).T

,useful_camera_number,main,camera_number,ultrawide,third,third_mp,fourth,fourth_mp,has_ultrawide,has_third_camera,has_fourth_camera
1 camera,1,50,2,NaN,Useless,2.0,None,None,False,True,False
"2 cameras, ultrawide",2,50,2,8,None,None,None,None,True,False,False
"2 cameras, no ultrawide",2,50,2,NaN,Telephoto - 2x,50.0,None,None,False,True,False
3 cameras,3,50,3,50,Telephoto - 3.5x,50.0,None,None,True,True,False
4 cameras,4,200,4,50,Telephoto - 5x,50.0,Telephoto - 3x,10.0,True,True,True


#### 9.3 A worked example

A mid-range phone with only a few inputs given; everything else is skipped:

In [ ]:
example_inputs = {
    "brand": "Samsung",
    "processor": "Mediatek Dimensity 7300",
    "display_type": "Amoled",
    "battery": 5000,
    "useful_camera_number": 3,
    "main": 50,
    "ultrawide": 8,
    "zoom": 3,
}

example = pm.predict_price(example_inputs, models, saved_metadata)

print(f"Predicted price: Rs. {example['prediction']:,.0f} (typical error ± Rs. {example['typical_error']:,.0f})")
print("\nTypical values used for skipped inputs:")
for label, value in example["filled"].items():
    print(f"  {label}: {value}")

Predicted price: Rs. 39,500 (typical error ± Rs. 6,538)

Typical values used for skipped inputs:
  Sales channel / tier: Online
  RAM type: LPDDR4X (typical for Mediatek Dimensity 7300)
  Storage type: UFS 2.2 (typical for Mediatek Dimensity 7300)
  Screen size (inches): 6.77 (typical for Mediatek Dimensity 7300)
  Curved display: False
  Refresh rate (Hz): 120
  Peak brightness (nits): 1300.0
  Screen protection: Corning, GG 7i
  Pixel density (ppi): 429
  Charging speed (W): 45
  Thickness (mm): 8.1 (typical for Mediatek Dimensity 7300)
  Back material: Plastic
  Frame material: Plastic
  Weight (g): 200
  Rear video: 4, 30
  Front camera (MP): 16.0
  Front video: 1, 30
  Speakers: Stereo
  Wireless charging: False
  IP rating: 68
  Launch date: today (2026-11-07)


#### 9.4 Performance with and without Geekbench, and for an unknown processor

In [ ]:
performance_examples = {
    "Known processor": {"processor": "Snapdragon 8 Elite"},
    "Known processor + Geekbench": {"processor": "Snapdragon 8 Elite", "geekbench_single": 2900, "geekbench_multiple": 9500},
    "Processor skipped, type given": {"processor_type": "Snapdragon", "fab": 3, "ram_type": "LPDDR5X", "rom_type": "UFS 4.0"},
}

rows = []

for label, inputs in performance_examples.items():
    result = pm.predict_performance(inputs, models, saved_metadata)
    rows.append({
        "case": label,
        "model": result["model"],
        "predicted AnTuTu": round(result["prediction"]),
        "typical error ±": round(result["typical_error"]),
    })

pd.DataFrame(rows)

,case,model,predicted AnTuTu,typical error ±
0,Known processor,Extra Trees (hardware),2593537,150570
1,Known processor + Geekbench,Extra Trees (hardware + Geekbench),2628472,108180
2,"Processor skipped, type given",Extra Trees (hardware),2879385,272469


For a skipped processor the typical error is the **processor-grouped** error from Section 4, because the model is then predicting for a chip it cannot identify. Notebook 04 showed that this is substantially less accurate than when the processor is known.

### 10. The App

The Streamlit app now has three pages:

| Page | What it does |
|---|---|
| Smartphone Finder | Recommendations from a sentence (Notebook 08) |
| Price Predictor | Estimates the price of a phone from its specifications |
| Performance Predictor | Estimates the AnTuTu score from processor and memory |

On the predictor pages:

- every input is a dropdown or a number box, and **leaving it empty skips it** (the typical value is used, and the page lists which values were filled in)
- choosing a processor fills in its type and fabrication size
- the camera questions follow the rule in Section 7.4
- the Performance Predictor uses the Geekbench model when both Geekbench scores are entered
- every estimate is shown with its typical error from cross-validation

Run it from the project root after this notebook has been run once:

```bash
streamlit run app/app.py
```

### 11. Limitations

- **Estimates, not market prices.** The price model learned from 289 phones; its typical error is about Rs. 6,500, and errors are larger for unusual or very expensive phones.
- **Skipped inputs pull estimates towards a typical phone.** The more inputs are skipped, the closer the estimate gets to the middle of the market.
- **Unseen categories are ignored.** Dropdowns only offer values from the dataset, but a processor that is not listed is unknown to the performance model, which then relies on processor type, fabrication, and memory.
- **Camera details beyond the asked inputs are typical values**, not the phone's real configuration.
- **Price and performance models are trained on all phones**, including discontinued ones; prices are those recorded in the dataset.